# PMXT uncertainty handling and label certification

Version 3 replays the same six cached receipt hours and three source analysis hours as version 2. It handles same-time updates atomically, retains quantity alternatives, isolates uncertainty by side/level, and certifies labels using economic bounds. It does not download data, train models, access a new final test or estimate net trading P&L.

See [the validation report](../docs/reports/pmxt_validation.md). Default execution verifies saved artifacts. Occurrence certification is different from complete book coverage: reliable expensive legs can rule out a basket despite an unknown member.

In [1]:
from pathlib import Path
import sys,json
import pandas as pd
from IPython.display import display
PROJECT=Path.cwd()
if PROJECT.name=='notebooks': PROJECT=PROJECT.parent
sys.path.insert(0,str(PROJECT))
from src.common import sha256
from src.data.pmxt_pilot import DEFAULT_ROOT,load_plan,verify
from src.data.pmxt_revalidation import rerun
ROOT=DEFAULT_ROOT
NAME='source_validation_v3'
OUTPUT=ROOT/NAME
RESULTS=PROJECT/'results/pmxt_pilot'/ROOT.name/NAME
REBUILD=False
if REBUILD:
    rerun(ROOT,name=NAME)
summary=json.loads((RESULTS/'summary.json').read_text())
assert sha256(OUTPUT/'design.json')==summary['design_sha256']
_,plan=load_plan(ROOT)
assert sha256(ROOT/'plan.json')==json.loads((OUTPUT/'design.json').read_text())['universe_sha256']
for item in summary['inputs']:
    assert sha256(ROOT/item['path'])==item['sha256'],item['path']
for path,digest in summary['implementation_sha256'].items():
    assert sha256(PROJECT/path)==digest,path
for path,digest in summary['derived_sha256'].items():
    assert sha256(OUTPUT/path)==digest,path
for path,digest in summary['results_sha256'].items():
    assert sha256(RESULTS/path)==digest,path
display(verify(ROOT))

{'unchanged': True, 'files': 2913}

In [2]:
display(pd.Series({k:summary[k] for k in [
    'source_start','source_end','receipt_end','events','tokens','event_hours',
    'full_book_event_hours','full_ask_event_hours','occurrence_known_hours',
    'occurrence_positive_hours','occurrence_negative_hours','occurrence_unknown_hours',
    'exact_value_hours','value_lower_bound_hours','full_ask_time_fraction',
    'occurrence_determined_time_fraction','arrival_checks','net_pnl_estimated']}))
comparison=pd.read_csv(RESULTS/'comparison_v2.csv')
display(pd.DataFrame({
    'v2':{'known_occurrence':comparison.v2_occurrence.notna().sum(),
          'positive':comparison.v2_occurrence.eq(1).sum(),
          'negative':comparison.v2_occurrence.eq(0).sum()},
    'v3':{'known_occurrence':comparison.label_occurrence.notna().sum(),
          'positive':comparison.label_occurrence.eq(1).sum(),
          'negative':comparison.label_occurrence.eq(0).sum()}}))

source_start                           2026-05-04 01:00:00+00:00
source_end                             2026-05-04 04:00:00+00:00
receipt_end                            2026-05-04 06:00:00+00:00
events                                                        15
tokens                                                       200
event_hours                                                   45
full_book_event_hours                                          7
full_ask_event_hours                                           8
occurrence_known_hours                                        24
occurrence_positive_hours                                      0
occurrence_negative_hours                                     24
occurrence_unknown_hours                                      21
exact_value_hours                                             24
value_lower_bound_hours                                        0
full_ask_time_fraction                                  0.554194
occurrence_determined_tim

,v2,v3
known_occurrence,7,24
positive,0,0
negative,7,24


In [3]:
hours=pd.read_csv(RESULTS/'event_hours.csv')
display(hours[['event_slug','hour','fully_observed_seconds','full_ask_seconds',
    'ruled_out_seconds','label_occurrence','peak_value_lower_usd','peak_value_upper_usd',
    'label_peak_value_usd','value_target_status']])
display(pd.Series(summary['raw_counters']).sort_values(ascending=False))
# Flags count messages/side groups, not independent outages.
display(pd.read_csv(RESULTS/'token_quality.csv'))

,event_slug,hour,fully_observed_seconds,full_ask_seconds,ruled_out_seconds,label_occurrence,peak_value_lower_usd,peak_value_upper_usd,label_peak_value_usd,value_target_status
0,databricks-ipo-closing-market-cap,2026-05-04 01:00:00+00:00,2707.461,3024.928,3600.000,0.0,0.0,0.0,0.0,exact
1,bitcoin-vs-gold-vs-sp-500-in-2026,2026-05-04 01:00:00+00:00,3600.000,3600.000,3600.000,0.0,0.0,0.0,0.0,exact
2,us-trade-deficit-in-2026,2026-05-04 01:00:00+00:00,1675.840,1941.486,3600.000,0.0,0.0,0.0,0.0,exact
3,how-many-tornadoes-in-the-us-in-2026,2026-05-04 01:00:00+00:00,3086.984,3534.671,3600.000,0.0,0.0,0.0,0.0,exact
4,colombia-presidential-election-1st-round-turnout,2026-05-04 01:00:00+00:00,3415.575,3415.575,3600.000,0.0,0.0,0.0,0.0,exact
5,chi1-edv-ccu-2026-05-22,2026-05-04 01:00:00+00:00,3600.000,3600.000,3600.000,0.0,0.0,0.0,0.0,exact
6,openai-ipo-closing-market-cap-554,2026-05-04 01:00:00+00:00,2824.000,2824.000,3600.000,0.0,0.0,0.0,0.0,exact
7,india-annual-inflation-2026,2026-05-04 01:00:00+00:00,2515.809,2515.809,3600.000,0.0,0.0,0.0,0.0,exact
8,zelenskyy-of-tweets-may-5-may-12-2026,2026-05-04 01:00:00+00:00,425.147,1013.373,3527.805,NaN,0.0,NaN,NaN,unknown_interval
9,of-seats-won-by-ppp-in-south-korea-by-elections,2026-05-04 01:00:00+00:00,1801.127,3261.149,3261.149,NaN,0.0,NaN,NaN,unknown_interval


price_change                                1153506
best_price_contradiction                     123012
identical_level_replays                       48731
book                                          33780
compatible_snapshot_close_groups              16443
identical_snapshot_replays                    10049
best_price_compatible_with_bounded_depth       7500
unresolved_level_conflicts                     1550
last_trade_price                               1478
level_conflicts_resolved_by_later_update       1427
snapshot_side_recoveries                       1024
unresolved_snapshot_level_ties                   12
tick_size_change                                  5
level_conflicts_resolved_by_snapshot              2
dtype: int64

,asset_id,price_change,identical_level_replays,book,identical_snapshot_replays,snapshot_side_recoveries,compatible_snapshot_close_groups,best_price_contradiction,last_trade_price,unresolved_level_conflicts,best_price_compatible_with_bounded_depth,level_conflicts_resolved_by_later_update,unresolved_snapshot_level_ties,level_conflicts_resolved_by_snapshot,tick_size_change
0,1013303232772613078089364769282479364842719188...,8909,352.0,143,13,9,82,1363.0,9.0,NaN,NaN,NaN,NaN,NaN,NaN
1,1025205157286985031646867846747978044328792260...,5947,263.0,136,16,4,87,344.0,2.0,1.0,1.0,1.0,NaN,NaN,NaN
2,1028431895329686584841247243841691454310968435...,4513,NaN,105,1,3,16,18.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN
3,1030857673646428597599568112983351459583344748...,7841,370.0,159,12,11,95,802.0,8.0,9.0,39.0,9.0,NaN,NaN,NaN
4,1031029027005737910334429611245959257202457300...,1173,64.0,212,72,0,127,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,9779779365142063138759332447581984349844925758...,1002,2.0,141,60,0,63,NaN,7.0,NaN,NaN,NaN,NaN,NaN,NaN
196,9841071128013801426343304957794320306607184788...,11380,536.0,94,1,12,54,619.0,NaN,1.0,21.0,NaN,NaN,NaN,NaN
197,9865378754846136456030025275933434374337253405...,493,26.0,251,165,0,78,NaN,3.0,NaN,NaN,NaN,NaN,NaN,NaN
198,9943734038648994121075062636841914198681847773...,5028,230.0,154,26,4,91,119.0,2.0,18.0,29.0,18.0,NaN,NaN,NaN


In [4]:
features=pd.read_parquet(OUTPUT/'hourly_features.parquet')
labels=pd.read_parquet(OUTPUT/'hourly_label_bounds.parquet')
assert not features.duplicated(['event_id','hour']).any()
assert not labels.duplicated(['event_id','hour']).any()
assert labels.loc[labels.label_occurrence==0,'ruled_out_seconds'].ge(3600-1e-8).all()
assert labels.loc[labels.label_occurrence==1,'peak_value_lower_usd'].gt(0).all()
exact=labels.label_peak_value_usd.notna()
assert (labels.loc[exact,'peak_value_upper_usd']-labels.loc[exact,'peak_value_lower_usd']).abs().le(1e-8).all()
assert labels.loc[labels.value_target_status=='lower_bound','label_peak_value_usd'].isna().all()
print('Occurrence and censored-value invariants passed.')
print(summary['feature_availability'])
checks=pd.read_csv(RESULTS/'arrival_checks.csv')
display(checks)
if checks.empty:
    print('No accepted candidate episode starts: empirical latency comparison unavailable.')
display(pd.Series({k:summary[k] for k in ['historical_parameters_verified','models_trained',
    'new_final_test_accessed','raw_cache_bytes','limitations']}))

Occurrence and censored-value invariants passed.
Causal source-time states before scan; later retrospective masks excluded from model inputs.


,event_id,package_id,decision_at,arrival_at,latency_ms,quantity,quoted_gross_usd,status,conditional_gross_usd


No accepted candidate episode starts: empirical latency comparison unavailable.


historical_parameters_verified                                                False
models_trained                                                                False
new_final_test_accessed                                                       False
raw_cache_bytes                                                            13580838
limitations                       [Source-ms batch completion and compatible clo...
dtype: object

Features describe causal source-time states strictly before each scan. Later drift findings can exclude labels, but do not enter the model inputs. A positive requires a candidate supported by the retained alternatives; a negative requires every basket to be ruled out for the full hour. A partial hour's observed maximum is a lower bound unless its upper bound matches it.

All monetary values are conditional zero-fee displayed margins under the recorded minimum/tick scenarios. Arrival checks use fixed quantities and limits, with 100 ms primary and 0/50/250/500 ms sensitivity. They are not a complete top-N policy replay, actual fills or net P&L. Previous PMXT artifacts and Dome results are preserved.